# Model info, cache status and loading progress

[Back to the introduction](ch01_demo.ipynb). Run in order. `LLM.model_info()` lists the files a model needs, their sizes and whether the Transformers.js browser cache already holds them, without downloading weights. `progress_callback` reports loading events to Python.

`download_bytes` is the download size, not runtime memory. `cache_status` only reflects the Transformers.js Cache API: hosted JupyterLite files and ZIPs in `/drive/models` are not counted. Unknown values are `None` or `"unknown"`, never guessed. An explicit `dtype` lets the check work offline.

In [ ]:
from lmlite import LLM, cached_models, clear_cache

model_id = "Xenova/distilgpt2"
dtype = "q8"


def show(info):
    total = "unknown" if info.download_bytes is None else f"{info.download_bytes / 1e6:.1f} MB"
    print(f"{info.model_id} ({info.task}, {info.backend}, {info.dtype}): cache={info.cache_status}, download={total}")
    for file in info.files:
        size = "?" if file["size"] is None else f"{file['size'] / 1e6:.2f} MB"
        print(f"  {'cached' if file['cached'] else 'remote':6} {size:>9}  {file['name']}")
    if info.error:
        print("  error:", info.error)

In [ ]:
show(await LLM.model_info(model_id, backend="wasm", dtype=dtype))

`cache_status` does not cover hosted files. `LLM.hosted_base()` checks the site's `files/models/manifest.json` (or `hosted_models_url=`) and returns the base URL serving the model, or `None` when `LLM.create()` would use the browser cache or the Hub instead. It downloads no model files.

In [ ]:
base = await LLM.hosted_base(model_id)
print(f"{model_id}: hosted at {base}" if base else f"{model_id}: not hosted; loads from the browser cache or the Hub")

In [ ]:
def on_progress(event):
    # Print file completions and status changes, not every byte update.
    if event["status"] != "progress" or (event["total"] and event["loaded"] == event["total"]):
        print(event)


llm = await LLM.create(generator_model=model_id, backend="wasm", dtype=dtype, progress_callback=on_progress)

In [ ]:
show(await LLM.model_info(model_id, backend="wasm", dtype=dtype))
await llm.dispose()

## Cache manager

`lmlite.widgets.CacheManager` is a widget built on `cached_models()`, `clear_cache()` and `LLM.create()`. One line per model in the Transformers.js browser cache: architecture (from the cached `config.json`), task (from the Hub, `?` offline), name and size, with a button to delete it (all revisions). Click a model name to expand the list of its cached files and their sizes. Enter a model ID and click Load to download it into the cache; it is loaded and released right away. With "auto", the task comes from the model's Hub tag; pick a task in the list to override it (needed offline or for models without a tag). Hosted model files and ZIPs under `/drive/models` are not listed.

In [ ]:
from IPython.display import display
from lmlite.widgets import CacheManager

manager = CacheManager()
display(manager.view)
await manager.refresh()

Disposal keeps cached files. Clearing the cache removes every Transformers.js download, so the next load downloads again.

In [ ]:
print(await clear_cache())
show(await LLM.model_info(model_id, backend="wasm", dtype=dtype))